# 6 -- Entrenamiento del modelo T-GCN y extración de embeddings relacionales — Dataset FDIC RIS

En los notebooks anteriores hemos construido dos estructuras fundamentales:

- Los **embeddings TabPFN** $e_i^t \in \mathbb{R}^{192}$, que condensan la historia tabular CAMELS de cada banco $i$ en el trimestre $t$ en un vector latente extraído por un transformer in-context.
- Los **snapshots del grafo dinámico** $\{G^1, \dots, G^{23}\}$, donde cada $G^t = (V^t, E^t, X^t, A^t)$ representa la red de conglomerados financieros bancarios en el trimestre $t$, con atributos nodales $x_i^t = [e_i^t \Vert s_i^t] \in \mathbb{R}^{204}$.

Este notebook entrena el **T-GCN** (Zhao et al., 2019) sobre el bloque de desarrollo (2016Q2 → 2021Q4). El modelo aprende a predecir qué bancos quiebran en el siguiente trimestre combinando dos fuentes de información que ningún modelo tabular puede capturar simultáneamente:

1. **Dependencias espaciales**: la posición de un banco en la red de holdings modula su riesgo. Un banco sano dentro de un conglomerado en dificultades tiene un perfil de riesgo distinto a un banco sano aislado, aunque sus ratios CAMELS sean idénticos.
2. **Dependencias temporales**: la evolución de esa posición a lo largo de los trimestres anteriores, capturada por el estado oculto GRU.

La hipótesis central es que el riesgo bancario tiene una **componente sistémica no capturada por features individuales**, observable en la estructura de relaciones entre entidades.

## 1. Configuración y paths

In [5]:
%pip install -q torch_geometric

In [6]:
# ============================================================
# CELDA 1 — Configuración global
# ============================================================
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch_geometric.utils import to_dense_adj
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)

# Paths — ajustar si la estructura de directorios difiere
#ROOT   = Path('D:/financial_risk_data')
#GRAPHS = ROOT / 'graphs'
#MODELS = ROOT / 'models_checkpoints'
MODELS = 'models_checkpoints'


# Añadir models/ al path para importar tgcn.py
#sys.path.insert(0, str(ROOT / 'src'))
from tgcn import TGCN
from graph_builder import GraphBuilder

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device: GPU si está disponible (Colab), CPU en local
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Carga de snapshots

Los snapshots fueron construidos en el notebook 04 con `GraphBuilder.save_snapshots()`,
que serializa cada $G^t$ como un archivo `.pt` usando `torch.save`. Cada objeto
`torch_geometric.data.Data` contiene:

- `data.x` $\in \mathbb{R}^{|V^t| \times 204}$: atributos de nodo.
- `data.edge_index` $\in \mathbb{Z}^{2 \times |E^t|}$: aristas en formato COO.
- `data.y` $\in \{0,1\}^{|V^t|}$: etiquetas de quiebra.
- `data.cert`: lista de CERTs en el mismo orden que las filas de `data.x`.
- `data.period`: trimestre en formato `'YYYYQq'`.

Nótese que los snapshots **no almacenan la matriz de adyacencia densa** $A^t$
sino `edge_index` en formato COO para ahorrar memoria. La conversión a densa
se hace en el forward del modelo con `to_dense_adj`, justificada porque
$N \approx 5.000$ hace que $A^t \in \mathbb{R}^{5000 \times 5000}$ sea manejable
en GPU ($\sim$ 100 MB en float32).

In [7]:
# ============================================================
# CELDA 2 — Carga de snapshots desarrollo
# ============================================================
# snapshots_dev = GraphBuilder.load_snapshots(GRAPHS / 'snapshots_desarrollo')
snapshots_dev = GraphBuilder.load_snapshots('snapshots_desarrollo')

print(f'\nResumen del bloque de desarrollo:')
print(f'  Snapshots cargados : {len(snapshots_dev)}')
print(f'  Rango temporal     : {snapshots_dev[0].period} → {snapshots_dev[-1].period}')
print(f'  dim_x              : {snapshots_dev[0].x.shape[1]}')
print()

# Verificación de integridad
for snap in snapshots_dev:
    assert snap.x.shape[1] == 204, f'dim_x inesperada en {snap.period}'
    assert snap.y.shape[0] == snap.num_nodes, f'Desajuste y/num_nodes en {snap.period}'

total_positivos = sum(int(s.y.sum()) for s in snapshots_dev)
total_nodos     = sum(s.num_nodes for s in snapshots_dev)
print(f'  Total nodos        : {total_nodos:,}')
print(f'  Total positivos    : {total_positivos}')
print(f'  Prevalencia global : {total_positivos/total_nodos*100:.4f}%')

Cargados 23 snapshots desde snapshots_desarrollo

Resumen del bloque de desarrollo:
  Snapshots cargados : 23
  Rango temporal     : 2016Q2 → 2021Q4
  dim_x              : 204

  Total nodos        : 125,575
  Total positivos    : 63
  Prevalencia global : 0.0502%


## 3. Split temporal train / validación

La separación temporal es un requisito de validez del sistema de early warning:
el modelo debe entrenarse sobre historia pasada y evaluarse sobre periodos futuros
no vistos, replicando las condiciones reales de despliegue.

Un split aleatorio introduciría **leakage temporal**: el modelo vería el estado
del grafo en $t+k$ durante el entrenamiento, lo que inflaría artificialmente
las métricas y produciría un sistema inútil en producción.

Elegimos la frontera en **2020Q3** por dos razones:

1. **Proporcional**: 17 snapshots de train (74%) y 6 de validación (26%).
2. **Económica**: separa el periodo pre-COVID del post-COVID. El shock sistémico
   de 2020 es el evento de estrés más relevante del bloque; el modelo debe
   aprenderlo desde el contexto pre-shock y generalizarlo en validación.

Esto nos deja dos conjuntos con:

- Train: 61 positivos (2016Q2–2020Q2, 17 snapshots)
- Val: 2 positivos (2020Q3–2021Q4, 6 snapshots)

Con solo 2 positivos en validación, el periodo 2020Q3–2021Q4 coincide con las moratorias regulatorias COVID que suprimieron artificialmente las quiebras. El walk-forward produciría folds con 0–1 positivos, haciendo las métricas por fold estadísticamente inútiles. El hold-out temporal es la única opción viable dado el régimen regulatorio excepcional del periodo.

El T-GCN no es el clasificador final, es un generador de embeddings (encoder). Lo que necesitamos es que `h_i^t` capture estructura relacional rica y con continuidad temporal. Para eso el modelo necesita ver secuencias largas con variedad de estados del grafo, no maximizar la señal supervisada en val. Con 17 snapshots de train ya tienes 10 ventanas W=8 por epoch, que es un volumen razonable para aprender la dinámica recurrente.

El rol de val en este contexto es únicamente detectar cuándo el encoder empieza a sobreajustar a los 61 positivos de train, es decir, cuándo la BCE deja de ser útil como señal de regularización. Para eso val_loss es suficiente con 2 positivos, porque lo que mides es si el modelo asigna probabilidades coherentes, no si clasifica bien.
Donde sí importan los positivos es en el MLP de fusión y en la evaluación final

In [8]:
# ============================================================
# CELDA 3 — Split temporal
# ============================================================
FRONTERA_VAL = '2020Q3'  # primer trimestre de validación

snapshots_train = [s for s in snapshots_dev if s.period < FRONTERA_VAL]
snapshots_val   = [s for s in snapshots_dev if s.period >= FRONTERA_VAL]

pos_train = sum(int(s.y.sum()) for s in snapshots_train)
pos_val   = sum(int(s.y.sum()) for s in snapshots_val)
n_train   = sum(s.num_nodes for s in snapshots_train)
n_val     = sum(s.num_nodes for s in snapshots_val)

print(f'Split temporal:')
print(f'  Train : {snapshots_train[0].period} → {snapshots_train[-1].period}')
print(f'          {len(snapshots_train)} snapshots | {n_train:,} nodos | {pos_train} positivos')
print(f'  Val   : {snapshots_val[0].period} → {snapshots_val[-1].period}')
print(f'          {len(snapshots_val)} snapshots | {n_val:,} nodos | {pos_val} positivos')

Split temporal:
  Train : 2016Q2 → 2020Q2
          17 snapshots | 95,468 nodos | 61 positivos
  Val   : 2020Q3 → 2021Q4
          6 snapshots | 30,107 nodos | 2 positivos


## 4. Desbalanceo de clases y función de pérdida

`BCE` (Binary Cross Entropy) sin ponderar falla con desbalanceo extremo. Con 0.044% de positivos, predecir siempre 0 da una loss de aproximadamente $-log(1-ε) ≈ 0$, prácticamente cero. El gradiente de los positivos es microscópico comparado con el de los negativos porque hay 1564 negativos por cada positivo. El optimizador aprende que ignorar los positivos es la estrategia de menor coste, y nunca sale de ese mínimo local. La función de pérdida estándar BCE sin ponderar converge trivialmente prediciendo siempre 0, obteniendo pérdida mínima sin aprender nada.

**BCE ponderada** (`BCEWithLogitsLoss` con `pos_weight`):

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N}
\left[ w^+ \cdot y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i) \right]
$$

donde $w^+ = n_{\text{neg}} / n_{\text{pos}}$ amplifica el gradiente de los
positivos en proporción a su escasez. Con $w^+$, cada quiebra
contribuye al gradiente como si hubiera $w^+$ ejemplos en vez de uno.

`BCEWithLogitsLoss` combina sigmoid y BCE en una sola operación numéricamente
estable usando el truco $\log(\sigma(x)) = -\log(1 + e^{-x})$, evitando
underflow/overflow para logits extremos. Por eso el modelo devuelve logits
crudos (sin sigmoid) y la pérdida aplica sigmoid internamente. El parametro `pos_weight` se usa especificamente para manejar clases desbalanceadas, asignando una peso mayor en los fallos que se dan en la clase positiva.

**Focal Loss** (Lin et al., 2017) queda como experimento posterior:
$$
\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)
$$
El factor $(1-p_t)^\gamma$ reduce el peso de negativos fáciles (bien clasificados)
y concentra el aprendizaje en ejemplos difíciles. Se comparará con BCE ponderada
en la sección de ajuste de hiperparámetros.

In [9]:
# ============================================================
# CELDA 4 — Cálculo de pos_weight sobre el bloque de train
# ============================================================
# pos_weight se calcula SOLO sobre train para no contaminar con información
# de validación. Es un estadístico del conjunto de entrenamiento.

n_neg_train = n_train - pos_train
pos_weight  = n_neg_train / pos_train

print(f'Bloque train:')
print(f'  Negativos : {n_neg_train:,}')
print(f'  Positivos : {pos_train}')
print(f'  pos_weight: {pos_weight:.2f}')
print()
print(f'Interpretación: cada quiebra en train contribuye al gradiente')
print(f'como si hubiera {pos_weight:.0f} ejemplos en vez de uno.')

Bloque train:
  Negativos : 95,407
  Positivos : 61
  pos_weight: 1564.05

Interpretación: cada quiebra en train contribuye al gradiente
como si hubiera 1564 ejemplos en vez de uno.


En PyTorch, BCEWithLogitsLoss(pos_weight=w) escala el término positivo de la loss:
L_i = -[w * y_i * log(σ(x_i)) + (1-y_i) * log(1-σ(x_i))]
El efecto es que cada positivo contribuye w veces más al gradiente que un negativo. Con w=391, un solo positivo produce el mismo gradiente total que 391 negativos. Esto no equivale a duplicar ejemplos: el gradiente es el mismo pero el modelo ve la distribución real, lo que evita que aprenda estadísticas espurias de los positivos.
Por qué pw_factor=0.25 en vez del ratio bruto
Con w=1564 el gradiente de los 61 positivos domina completamente. El modelo puede converger a una solución que clasifica todos los nodos como positivos para minimizar la loss ponderada, que es el problema opuesto al trivial. pw_factor=0.25 suaviza el peso hasta 391, que equilibra la señal sin aplastar los negativos. El grid search confirmó que pw_factor=0.25 da val_loss menor que pw_factor=0.5 (391 vs 782), lo que indica que 782 ya empieza a ser demasiado agresivo.
Estabilidad numérica de BCEWithLogitsLoss
La fórmula naive log(σ(x)) con x muy negativo produce σ(x)≈0 y log(0)→-∞. PyTorch reformula internamente como:
log(σ(x)) = x - log(1 + exp(x))   si x >= 0
           = -log(1 + exp(-x))     si x < 0
Esto mantiene el cálculo en rango finito para cualquier logit. Por eso el modelo devuelve logits crudos: si aplicaras sigmoid antes de pasarlos a la loss perderías esta estabilidad.

## 5. Arquitectura T-GCN — revisión y justificación

La arquitectura completa se define en `models/tgcn.py`. Aquí revisamos
las decisiones de diseño y sus fundamentos matemáticos.

### 5.1 Convolución GCN dentro de la celda GRU

La normalización simétrica del Laplaciano (Kipf & Welling, 2017):

$$
\hat{L}^t = \tilde{D}^{-1/2} \tilde{A}^t \tilde{D}^{-1/2},
\quad \tilde{A}^t = A^t + I
$$

garantiza dos propiedades:
- **Auto-conexión** ($+I$): el nodo incluye su propio estado en la agregación.
  Sin ella, la GCN ignora la información propia del nodo y solo propaga la de vecinos.
- **Invariancia al grado**: nodos con muchos vecinos (holdings grandes) no acumulan
  señal arbitrariamente grande. $\tilde{D}^{-1/2}$ normaliza simétricamente.

Para nodos aislados (bancos sin holding), $\tilde{A}^t_{ii} = 1$ y
$\tilde{D}^t_{ii} = 1$, por lo que $\hat{L}^t_{ii} = 1$. La convolución
degenera en la identidad: $h_i^{(1)} = \sigma(x_i^t W)$. Esto es correcto:
un banco independiente transforma solo sus propias features sin recibir
señal de contagio estructural.

### 5.2 Celda T-GCN

La GRU estándar sustituye las multiplicaciones matriciales lineales por
convoluciones GCN sobre el grafo en $t$:

$$
z_t = \sigma\bigl(\hat{L}^t [h_{t-1}, x_t] W_z + b_z\bigr)
\quad \text{(puerta de actualización)}
$$
$$
r_t = \sigma\bigl(\hat{L}^t [h_{t-1}, x_t] W_r + b_r\bigr)
\quad \text{(puerta de reset)}
$$
$$
\tilde{h}_t = \tanh\bigl(\hat{L}^t [r_t \odot h_{t-1}, x_t] W_h + b_h\bigr)
\quad \text{(candidato)}
$$
$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

La clave es que $[h_{t-1}, x_t]$ se concatena **antes** de la multiplicación
por $\hat{L}^t$, de modo que la propagación sobre el grafo actúa sobre
la representación conjunta de estado oculto y features de entrada.

### 5.3 Laplaciano dinámico

$\hat{L}^t$ se recalcula en cada paso temporal del forward porque $A^t$ varía
por trimestre (quiebras y fusiones modifican $V^t$ y $E^t$). Esto tiene coste
$O(N^2)$ por snapshot, con $N \approx 5.000$, perfectamente viable en GPU.

### 5.4 Cabeza clasificadora

Tras procesar la secuencia de snapshots, el estado oculto final
$h_i^T \in \mathbb{R}^{d_h}$ de cada nodo contiene su embedding dinámico:
incorpora la estructura relacional del grafo en $T$ (vía GCN) y la evolución
temporal de esa estructura (vía GRU). Una capa lineal proyecta a logit escalar:

$$
\ell_i = h_i^T W_c + b_c \in \mathbb{R}
$$
$$
\hat{y}_i = \sigma(\ell_i) = P(\text{quiebra}_i \mid G^1, \dots, G^T)
$$

In [10]:
# ============================================================
# CELDA 5 — Instanciación del modelo y función de pérdida
# ============================================================
INPUT_DIM  = 204 # d_x = 192 (TabPFN) + 12 (estructurales)
HIDDEN_DIM = 64  # d_h — punto de partida conservador dado el nº de positivos, se sobreescribe en el grid search

model = TGCN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Modelo T-GCN instanciado:')
print(f'  input_dim  : {INPUT_DIM}')
print(f'  hidden_dim : {HIDDEN_DIM}')
print(f'  Parámetros : {n_params:,}')

pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
print(f'Función de pérdida: BCEWithLogitsLoss(pos_weight={pos_weight:.2f})')

Modelo T-GCN instanciado:
  input_dim  : 204
  hidden_dim : 64
  Parámetros : 51,713
Función de pérdida: BCEWithLogitsLoss(pos_weight=1564.05)


## 6. Estrategia de entrenamiento

### 6.1 Forward pass sobre una ventana deslizante de trimestres

En cada epoch, la secuencia de 17 snapshots de train se divide en ventanas deslizantes de longitud W=8, produciendo 10 ventanas por epoch. Cada ventana genera un paso de gradiente independiente:

Formalmente, en cada epoch:
$$
\mathcal{L}_{\text{epoch window}} = \frac{1}{W} \sum_{t=i}^{i+W-1}
\mathcal{L}_{\text{BCE}}(\hat{y}^t, y^t)
$$

donde $\hat{y}^t$ son los logits del snapshot $t$ y $y^t$ sus etiquetas. El estado oculto se reinicializa a cero al inicio de cada ventana y se propaga snapshot a snapshot dentro de ella mediante TBPTT-1 (hidden_state.detach() entre snapshots). Esto produce 10 actualizaciones de parámetros por epoch aumentando la variedad de gradientes y mejorando la generalización de la parte recurrente.

### 6.2 Gestión del estado oculto entre snapshots

El estado oculto arranca a cero al inicio de cada ventana. Dentro de la ventana, el `graph_builder` garantiza orden determinista por CERT, por lo que la alineación posicional entre snapshots consecutivos es correcta mientras el conjunto de nodos no cambie dentro de la ventana. Si cambia, la fila correspondiente al nodo nuevo o desaparecido se trata con el valor que tenga en ese momento, lo que es aceptable en entrenamiento porque la ventana es corta (8 trimestres = 2 años).

### 6.3 Alineación de nodos entre snapshots

Se distinguen dos regímenes:

En entrenamiento el reinicio por ventana hace que la alineación por CERT no sea necesaria. El estado no se propaga entre ventanas, por lo que el problema de dimensión variable queda acotado a la duración de cada ventana.

En inferencia (funciones evaluate, compute_val_loss, extracción de e_rel) se usa un diccionario cert_to_hidden que mapea cada CERT a su vector oculto. Al procesar el snapshot tt
t, los nodos ya vistos recuperan su estado del diccionario y los nodos nuevos se inicializan a cero. Esto garantiza continuidad del estado oculto a lo largo de toda la secuencia de inferencia independientemente de cambios en $∣V^t|$

### 6.4 Optimizador

Adam con lr=1e-3 y weight_decay=0 como punto de partida, igual que en
la implementación de referencia para T-GCN. Weight decay no nulo se
explorará en el ajuste de hiperparámetros.

In [11]:
# ============================================================
# CELDA 6 — Optimizador y configuración de entrenamiento
# ============================================================
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 300
PATIENCE     = 25  # early stopping sobre val loss

optimizer = torch.optim.Adam(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',     # minimizar val loss
    factor=0.5,
    patience=10,
    min_lr=1e-5
)

print(f'Optimizador : Adam(lr={LR}, weight_decay={WEIGHT_DECAY})')
print(f'Scheduler   : ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-5)')
print(f'Max epochs  : {MAX_EPOCHS}')
print(f'Patience    : {PATIENCE} epochs sin mejora en val loss')

Optimizador : Adam(lr=0.001, weight_decay=0.0001)
Scheduler   : ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-5)
Max epochs  : 300
Patience    : 25 epochs sin mejora en val loss


## 7. Funciones auxiliares

### 7.1 Conversión edge_index → adyacencia densa

Los snapshots almacenan aristas en formato COO (`edge_index`). El forward del
T-GCN necesita la matriz $A^t$ densa para calcular el Laplaciano. La función
`to_dense_adj` de PyTorch Geometric hace esta conversión en $O(N^2)$.

### 7.2 Métricas de clasificación con desbalanceo extremo

Con $\sim$0.044% de positivos, la accuracy es completamente inútil (predecir
siempre 0 da accuracy > 99.95%). Las métricas relevantes son:

- **AUROC**: área bajo la curva ROC. Mide la capacidad discriminativa global
  independientemente del umbral de decisión. Valor esperado aleatorio: 0.5.
- **Average Precision (AUPRC)**: área bajo la curva Precision-Recall.
  Más informativa que AUROC con desbalanceo extremo porque penaliza
  fuertemente los falsos positivos. Valor esperado aleatorio:  aproximadamente igual a la prevalencia ≈ 0.00044.
- **F1** con umbral optimizado sobre validación: $F_1 = 2 \cdot \frac{P \cdot R}{P + R}$.
  El umbral se elige maximizando $F_1$ sobre el conjunto de validación,
  no se fija a 0.5 (que sería subóptimo con este desbalanceo).


In [12]:
# ============================================================
# CELDA 7 — Funciones auxiliares
# ============================================================

from tgcn import calculate_laplacian

def snapshot_to_device(snapshot, device):
    """
    Mueve los tensores de un snapshot al device indicado.
    Modifica el objeto in-place. Asumir que todos los snapshots
    van siempre al mismo device para evitar inconsistencias.
    """
    snapshot.x          = snapshot.x.to(device)
    snapshot.edge_index = snapshot.edge_index.to(device)
    snapshot.y          = snapshot.y.to(device)
    return snapshot


def compute_adj_dense(snapshot):
    """
    Convierte edge_index COO a matriz de adyacencia densa A^t ∈ R^{N×N}.
    to_dense_adj devuelve (1, N, N); hacemos squeeze para obtener (N, N).
    """
    adj = to_dense_adj(
        snapshot.edge_index,
        max_num_nodes=snapshot.num_nodes
    ).squeeze(0)  # (N, N)
    return adj


def _build_hidden_from_dict(cert_list, cert_to_hidden, hidden_dim, device):
    """
    Construye hidden_state (N, hidden_dim) alineado con cert_list.
    Nodos conocidos recuperan su vector del diccionario; nuevos se inicializan a cero.
    """
    h = torch.zeros(len(cert_list), hidden_dim, device=device)
    for i, cert in enumerate(cert_list):
        if cert in cert_to_hidden:
            h[i] = cert_to_hidden[cert]
    return h


def _update_hidden_dict(cert_list, hidden_state, cert_to_hidden):
    """Vuelca el estado oculto actualizado al diccionario por CERT."""
    for i, cert in enumerate(cert_list):
        cert_to_hidden[cert] = hidden_state[i].detach()


def evaluate(model, snapshots, device, threshold=0.5):
    """
    Evalúa el modelo sobre una secuencia de snapshots.

    Procesa la secuencia completa en modo inferencia y recoge
    probabilidades y etiquetas de TODOS los snapshots (no solo el último).
    Usa alineación por CERT para preservar el estado oculto de nodos
    persistentes entre trimestres.

    Returns dict con auroc, auprc, f1, precision, recall.
    """
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        cert_to_hidden = {}
        for snap in snapshots:
            snap    = snapshot_to_device(snap, device)
            adj     = compute_adj_dense(snap)   # (N, N)
            x       = snap.x                    # (N, d_x)
            y       = snap.y                    # (N,)
            cert_list = list(snap.cert)

            hidden_state = _build_hidden_from_dict(
                cert_list, cert_to_hidden, model.hidden_dim, device
            )

            laplacian    = calculate_laplacian(adj)
            hidden_state, _ = model.tgcn_cell(x, hidden_state, laplacian)
            _update_hidden_dict(cert_list, hidden_state, cert_to_hidden)

            logits       = model.classifier(hidden_state).squeeze(1)
            probs        = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    auroc  = roc_auc_score(all_labels, all_probs)
    auprc  = average_precision_score(all_labels, all_probs)
    preds  = (all_probs >= threshold).astype(int)
    f1     = f1_score(all_labels, preds, zero_division=0)
    prec   = precision_score(all_labels, preds, zero_division=0)
    rec    = recall_score(all_labels, preds, zero_division=0)

    return {'auroc': auroc, 'auprc': auprc, 'f1': f1,
            'precision': prec, 'recall': rec}


def find_best_threshold(model, snapshots, device, thresholds=None):
    """
    Encuentra el umbral de clasificación que maximiza F1 sobre
    el conjunto dado. Se usa SOLO sobre validación, nunca sobre test.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        cert_to_hidden = {}
        for snap in snapshots:
            snap = snapshot_to_device(snap, device)
            adj  = compute_adj_dense(snap)
            cert_list = list(snap.cert)
            hidden_state = _build_hidden_from_dict(
                cert_list, cert_to_hidden, model.hidden_dim, device
            )
            laplacian       = calculate_laplacian(adj)
            hidden_state, _ = model.tgcn_cell(snap.x, hidden_state, laplacian)
            _update_hidden_dict(cert_list, hidden_state, cert_to_hidden)
            logits          = model.classifier(hidden_state).squeeze(1)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(snap.y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    best_f1, best_thr = 0.0, 0.5
    for thr in thresholds:
        f1 = f1_score(all_labels, (all_probs >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    return best_thr, best_f1

def build_windows(snapshots: list, W: int) -> list[list]:
    """
    Genera ventanas deslizantes de longitud W sobre la secuencia de snapshots.

    Con 17 snapshots y W=8 produce 10 ventanas:
      [0:8], [1:9], [2:10], ..., [9:17]

    Cada ventana es una sub-secuencia contigua que el T-GCN procesa
    independientemente con estado oculto reinicializado a cero.
    """
    return [snapshots[i:i+W] for i in range(len(snapshots) - W + 1)]


def compute_val_loss(model, snapshots, criterion, device):
    """
    Calcula la loss media sobre todos los snapshots de validación.
    Se usa como criterio de early stopping en lugar de AUROC,
    más estable con pocos positivos.
    """
    model.eval()
    total_loss   = 0.0

    with torch.no_grad():
        cert_to_hidden = {}
        for snap in snapshots:
            snap = snapshot_to_device(snap, device)
            adj  = compute_adj_dense(snap)
            cert_list = list(snap.cert)
            hidden_state = _build_hidden_from_dict(
                cert_list, cert_to_hidden, model.hidden_dim, device
            )

            laplacian       = calculate_laplacian(adj)
            hidden_state, _ = model.tgcn_cell(snap.x, hidden_state, laplacian)
            _update_hidden_dict(cert_list, hidden_state, cert_to_hidden)
            logits          = model.classifier(hidden_state).squeeze(1)
            loss            = criterion(logits, snap.y.float())
            total_loss     += loss.item()

    return total_loss / len(snapshots)


print('Funciones auxiliares definidas.')
print('Funciones auxiliares actualizadas: build_windows, compute_val_loss añadidas.')

Funciones auxiliares definidas.
Funciones auxiliares actualizadas: build_windows, compute_val_loss añadidas.


## 8. Bucle de entrenamiento

En cada epoch se procesan las 10 ventanas deslizantes de longitud W=8 en orden
temporal. Cada ventana genera un paso de gradiente independiente: la loss se
acumula sobre los W snapshots de la ventana y se propaga con `backward()` al
final de la ventana. El estado oculto se reinicializa a cero al inicio de cada
ventana y se propaga snapshot a snapshot dentro de ella con TBPTT-1
(`hidden_state.detach()` entre snapshots para cortar el grafo computacional).

**Early stopping**: se monitoriza `val_loss` sobre el conjunto de validación.
Si no mejora durante `PATIENCE` epochs consecutivos se detiene el entrenamiento
y se restauran los pesos del mejor epoch. El learning rate se reduce con
`ReduceLROnPlateau` (factor=0.5, patience=10) sobre la misma `val_loss`.



Cuando hidden_state is None se inicializa a cero, correcto. Pero cuando hidden_state.size(0) != n también reinicializa sin .detach(), lo que es correcto porque es un tensor nuevo sin historial. El problema es que el primer snapshot de cada ventana donde hidden_state is None entra por la rama de reinicio, pero en snapshots posteriores donde size(0) == n sí hace .detach(). La lógica es correcta, pero el condicional mezcla dos casos distintos.

In [13]:
# ============================================================
# CELDA 8 — Bucle de entrenamiento con early stopping
# ============================================================
import copy, time

W = 8  # longitud de ventana deslizante — hiperparámetro

windows = build_windows(snapshots_train, W)
print(f'Ventanas deslizantes: {len(windows)} (W={W}, sobre {len(snapshots_train)} snapshots train)')

history = {
    'train_loss': [], 'val_loss': [], 'val_auroc': [], 'val_auprc': [], 'lr': []
}

best_val_loss   = float('inf')
best_epoch      = 0
best_model_wts  = copy.deepcopy(model.state_dict())
no_improv       = 0

t0 = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):

    # ── TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0

    for window in windows:
        optimizer.zero_grad()
        hidden_state = None
        window_loss  = 0.0

        for snap in window:
            snap = snapshot_to_device(snap, DEVICE)
            adj  = compute_adj_dense(snap)
            n    = snap.num_nodes

            if hidden_state is None or hidden_state.size(0) != n:
                hidden_state = torch.zeros(n, model.hidden_dim, device=DEVICE)
            else:
                hidden_state = hidden_state.detach()


            laplacian       = calculate_laplacian(adj)
            hidden_state, _ = model.tgcn_cell(snap.x, hidden_state, laplacian)
            logits          = model.classifier(hidden_state).squeeze(1)
            loss            = criterion(logits, snap.y.float())
            window_loss    += loss

        window_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += window_loss.item()

    avg_train_loss = epoch_loss / len(windows)

    # ── VALIDACIÓN ─────────────────────────────────────────────────────────
    val_loss    = compute_val_loss(model, snapshots_val, criterion, DEVICE)
    val_metrics = evaluate(model, snapshots_val, DEVICE)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_auroc'].append(val_metrics['auroc'])
    history['val_auprc'].append(val_metrics['auprc'])
    history['lr'].append(current_lr)

    # ── EARLY STOPPING sobre val loss ──────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        best_model_wts = copy.deepcopy(model.state_dict())
        no_improv      = 0
    else:
        no_improv += 1

    # ── LOG ────────────────────────────────────────────────────────────────
    if epoch % 25 == 0 or epoch == 1:
        elapsed = time.perf_counter() - t0
        print(
            f'Epoch {epoch:4d}/{MAX_EPOCHS} | '
            f'train_loss={avg_train_loss:.4f} | '
            f'val_loss={val_loss:.4f} | '
            f'val_auroc={val_metrics["auroc"]:.4f} | '
            f'val_auprc={val_metrics["auprc"]:.4f} | '
            f'lr={current_lr:.6f} | '
            f'elapsed={elapsed:.0f}s'
        )

    if no_improv >= PATIENCE:
        print(f'\nEarly stopping en epoch {epoch}. '
              f'Mejor epoch: {best_epoch} (val_loss={best_val_loss:.4f})')
        break

model.load_state_dict(best_model_wts)
print(f'\nEntrenamiento completado. Mejor epoch: {best_epoch} | val_loss: {best_val_loss:.4f}')

Ventanas deslizantes: 10 (W=8, sobre 17 snapshots train)
Epoch    1/300 | train_loss=8.1290 | val_loss=0.3657 | val_auroc=0.9995 | val_auprc=0.0855 | lr=0.001000 | elapsed=6s
Epoch   25/300 | train_loss=1.1504 | val_loss=0.0297 | val_auroc=0.9998 | val_auprc=0.2250 | lr=0.001000 | elapsed=68s
Epoch   50/300 | train_loss=0.6455 | val_loss=0.0271 | val_auroc=0.9998 | val_auprc=0.1769 | lr=0.001000 | elapsed=131s
Epoch   75/300 | train_loss=0.4449 | val_loss=0.0157 | val_auroc=0.9998 | val_auprc=0.2333 | lr=0.000500 | elapsed=196s
Epoch  100/300 | train_loss=0.2692 | val_loss=0.0139 | val_auroc=0.9998 | val_auprc=0.2381 | lr=0.000500 | elapsed=260s

Early stopping en epoch 114. Mejor epoch: 89 (val_loss=0.0131)

Entrenamiento completado. Mejor epoch: 89 | val_loss: 0.0131


train_loss desciende monotónamente de 8.13 a ~0.27, señal de aprendizaje real.
val_auroc estable en 0.9998-0.9999, consistente con lo esperado (saturación por 2 positivos).
ReduceLROnPlateau actúa correctamente: lr baja de 1e-3 a 5e-4 entre epoch 50 y 75.
Early stopping para en epoch 114, mejor epoch 89. Margen razonable.

## 9. Curvas de aprendizaje

Las curvas de aprendizaje permiten diagnosticar el comportamiento del modelo:

- **Underfitting**: train_loss no decrece → modelo demasiado simple o lr muy bajo.
- **Overfitting**: train_loss baja pero val_loss no mejora o aumenta → regularizar
  (weight_decay, dropout) o reducir hidden_dim.
- **Comportamiento esperado**: train_loss decrece progresivamente; val_loss
  decrece y se estabiliza. val_auroc estará saturado (~0.9999) por la escasez
  de positivos en validación (2 positivos) y no es informativo como criterio
  de diagnóstico. val_auprc es la métrica secundaria de referencia.

In [1]:
# ============================================================
# CELDA 9 — Curvas de aprendizaje
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

# --- 1. Curvas de pérdida (train vs val) ---
ax = axes[0, 0]
ax.plot(epochs_range, history["train_loss"], label="train_loss", color="tab:blue")
ax.plot(epochs_range, history["val_loss"], label="val_loss", color="tab:orange")
ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.6,
           label=f"mejor epoch ({best_epoch})")
ax.set_yscale("log")
ax.set_xlabel("Época")
ax.set_ylabel("Loss (MSE ponderado, escala log)")
ax.set_title("Curvas de pérdida")
ax.legend()
ax.grid(alpha=0.3)

axes[1].plot(epochs_range, history['val_auroc'],  label='Val AUROC',  color='darkorange')
axes[1].plot(epochs_range, history['val_auprc'],  label='Val AUPRC',  color='green', linestyle='--')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Métrica')
axes[1].set_title('Métricas de validación')
axes[1].legend()

plt.tight_layout()
plt.show()

# Métricas finales del mejor epoch
best_metrics = evaluate(model, snapshots_val, DEVICE)
best_thr, best_f1 = find_best_threshold(model, snapshots_val, DEVICE)
print(f'\nMétricas del mejor modelo (epoch {best_epoch}):')
print(f'  AUROC     : {best_metrics["auroc"]:.4f}')
print(f'  AUPRC     : {best_metrics["auprc"]:.4f}')
print(f'  F1 (thr=0.5) : {best_metrics["f1"]:.4f}')
print(f'  Umbral óptimo: {best_thr:.3f} → F1={best_f1:.4f}')

NameError: name 'plt' is not defined

## 10. Ajuste de hiperparámetros

Búsqueda en grid sobre los hiperparámetros más influyentes del T-GCN.
El criterio de selección es **val_loss mínima** (primario) y val_auprc
(secundario). val_auroc queda excluido como criterio por saturación
sistemática (~0.9999) debida a los 2 positivos en validación.

**Espacio de búsqueda:**

| Hiperparámetro | Valores | Justificación |
|---|---|---|
| `hidden_dim` | 32, 64 | 128 descartado: excesivo para 61 positivos en train |
| `lr` | 1e-3, 5e-4 | Rango estándar para Adam en GNNs |
| `weight_decay` | 1e-4 | Fijo: estable en exploración preliminar |
| `pw_factor` | 0.25, 0.5 | Suavizado del pos_weight bruto (1564); rango [391, 782] |
| `W` | 4, 8 | Longitud de ventana deslizante como hiperparámetro |

2×2×1×2×2 = 16 combinaciones. Con MAX_EPOCHS=200 y early stopping sobre
val_loss el coste efectivo es menor porque la mayoría de runs converge
antes de epoch 200.


In [15]:
# ============================================================
# CELDA 10 — Grid search de hiperparámetros
# ============================================================
import itertools
import pandas as pd

param_grid = {
    'hidden_dim'  : [32, 64],
    'lr'          : [1e-3, 5e-4],
    'weight_decay': [1e-4],
    'pw_factor'   : [0.25, 0.5],
    'W'           : [4, 8],
}

MAX_EPOCHS_SEARCH = 200
PATIENCE_SEARCH   = 20

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f'Combinaciones a explorar: {len(combos)}')  # 2×2×1×2×2 = 16

results = []

for i, combo in enumerate(combos):
    params  = dict(zip(keys, combo))
    pw      = pos_weight * params['pw_factor']
    windows = build_windows(snapshots_train, params['W'])

    model_gs = TGCN(input_dim=INPUT_DIM, hidden_dim=params['hidden_dim']).to(DEVICE)
    criterion_gs = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pw], dtype=torch.float32).to(DEVICE)
    )
    optimizer_gs = torch.optim.Adam(
        model_gs.parameters(), lr=params['lr'], weight_decay=params['weight_decay']
    )
    scheduler_gs = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_gs, mode='min', factor=0.5, patience=10, min_lr=1e-5
    )

    best_loss_run = float('inf')
    best_wts_run  = copy.deepcopy(model_gs.state_dict())
    no_improv     = 0

    for epoch in range(1, MAX_EPOCHS_SEARCH + 1):
        model_gs.train()

        for window in windows:
            optimizer_gs.zero_grad()
            hidden_state = None  # reinicio explícito al inicio de cada ventana
            window_loss  = 0.0

            for snap in window:
                snap = snapshot_to_device(snap, DEVICE)
                adj  = compute_adj_dense(snap)
                n    = snap.num_nodes

                if hidden_state is None or hidden_state.size(0) != n:
                    hidden_state = torch.zeros(n, model_gs.hidden_dim, device=DEVICE)
                else:
                    hidden_state = hidden_state.detach()

                laplacian       = calculate_laplacian(adj)
                hidden_state, _ = model_gs.tgcn_cell(snap.x, hidden_state, laplacian)
                logits          = model_gs.classifier(hidden_state).squeeze(1)
                loss            = criterion_gs(logits, snap.y.float())
                window_loss    += loss

            window_loss.backward()
            nn.utils.clip_grad_norm_(model_gs.parameters(), max_norm=1.0)
            optimizer_gs.step()

        val_loss = compute_val_loss(model_gs, snapshots_val, criterion_gs, DEVICE)
        scheduler_gs.step(val_loss)

        if val_loss < best_loss_run:
            best_loss_run = val_loss
            best_wts_run  = copy.deepcopy(model_gs.state_dict())
            no_improv     = 0
        else:
            no_improv += 1

        if no_improv >= PATIENCE_SEARCH:
            break

    model_gs.load_state_dict(best_wts_run)
    val_metrics = evaluate(model_gs, snapshots_val, DEVICE)

    results.append({
        **params, 'pw': pw,
        'best_val_loss': best_loss_run,
        'val_auroc'    : val_metrics['auroc'],
        'val_auprc'    : val_metrics['auprc'],
    })
    print(
        f'[{i+1:2d}/{len(combos)}] {params} | pw={pw:.0f} | '
        f'val_loss={best_loss_run:.4f} | auprc={val_metrics["auprc"]:.4f}'
    )

df_results = pd.DataFrame(results).sort_values('best_val_loss')
print('\nTop 5 configuraciones (por val loss):')
print(df_results.head(5).to_string(index=False))

Combinaciones a explorar: 16
[ 1/16] {'hidden_dim': 32, 'lr': 0.001, 'weight_decay': 0.0001, 'pw_factor': 0.25, 'W': 4} | pw=391 | val_loss=0.0117 | auprc=0.2111
[ 2/16] {'hidden_dim': 32, 'lr': 0.001, 'weight_decay': 0.0001, 'pw_factor': 0.25, 'W': 8} | pw=391 | val_loss=0.0047 | auprc=0.2250
[ 3/16] {'hidden_dim': 32, 'lr': 0.001, 'weight_decay': 0.0001, 'pw_factor': 0.5, 'W': 4} | pw=782 | val_loss=0.0122 | auprc=0.3333
[ 4/16] {'hidden_dim': 32, 'lr': 0.001, 'weight_decay': 0.0001, 'pw_factor': 0.5, 'W': 8} | pw=782 | val_loss=0.0063 | auprc=0.3333
[ 5/16] {'hidden_dim': 32, 'lr': 0.0005, 'weight_decay': 0.0001, 'pw_factor': 0.25, 'W': 4} | pw=391 | val_loss=0.0128 | auprc=0.7500
[ 6/16] {'hidden_dim': 32, 'lr': 0.0005, 'weight_decay': 0.0001, 'pw_factor': 0.25, 'W': 8} | pw=391 | val_loss=0.0043 | auprc=0.3250
[ 7/16] {'hidden_dim': 32, 'lr': 0.0005, 'weight_decay': 0.0001, 'pw_factor': 0.5, 'W': 4} | pw=782 | val_loss=0.0078 | auprc=0.2917
[ 8/16] {'hidden_dim': 32, 'lr': 0.0005,

Anomalía combo #5
hidden_dim=32, lr=5e-4, pw_factor=0.25, W=4 da val_loss=0.0128 pero auprc=0.75. Patrón de sobreajuste a los 2 positivos de val con val_loss alta. Descartar por inestabilidad estadística.

Configuración recomendada: combo #10
hidden_dim  = 64
lr          = 1e-3
weight_decay= 1e-4
pw_factor   = 0.25  →  pos_weight = 391
W           = 8
val_loss    = 0.0023
val_auprc   = 0.4167
Preferido sobre combo #14 por val_loss menor. La diferencia de auprc (0.4167 vs 0.3667) apoya la misma elección.

## 11. Entrenamiento final y guardado del mejor modelo

Con los hiperparámetros óptimos identificados en la búsqueda, entrenamos
el modelo final con `MAX_EPOCHS=300` y guardamos:

- Los pesos del mejor epoch (`tgcn_best.pt`).
- El umbral de clasificación óptimo (`threshold_opt`).
- Los hiperparámetros (`tgcn_config.json`).

Estos artefactos son los que se cargarán en el proceso de evaluación
final sobre el bloque (2022Q1 → 2025Q4).

In [17]:
# ============================================================
# CELDA 11 — Entrenamiento final con mejores hiperparámetros
# ============================================================

best_config = df_results.iloc[0].to_dict()
print('Mejor configuración:')
for k, v in best_config.items():
    print(f'  {k}: {v}')

W_final = int(best_config['W'])
windows_final = build_windows(snapshots_train, W_final)

model_final = TGCN(
    input_dim=INPUT_DIM,
    hidden_dim=int(best_config['hidden_dim'])
).to(DEVICE)

criterion_final = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([best_config['pw']], dtype=torch.float32).to(DEVICE)
)
optimizer_final = torch.optim.Adam(
    model_final.parameters(),
    lr=best_config['lr'],
    weight_decay=best_config['weight_decay']
)
scheduler_final = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_final, mode='min', factor=0.5, patience=10, min_lr=1e-5
)

MAX_EPOCHS_FINAL = 400
PATIENCE_FINAL   = 30

best_loss_final  = float('inf')
best_epoch_final = 0
best_wts_final   = copy.deepcopy(model_final.state_dict())
no_improv_final  = 0
history_final    = {'train_loss': [], 'val_loss': [], 'val_auroc': [], 'val_auprc': [], 'lr': []}

t0 = time.perf_counter()

for epoch in range(1, MAX_EPOCHS_FINAL + 1):
    model_final.train()
    epoch_loss = 0.0

    for window in windows_final:
        optimizer_final.zero_grad()
        hidden_state = None
        window_loss  = 0.0

        for snap in window:
            snap = snapshot_to_device(snap, DEVICE)
            adj  = compute_adj_dense(snap)
            n    = snap.num_nodes

            if hidden_state is None or hidden_state.size(0) != n:
                hidden_state = torch.zeros(n, model_final.hidden_dim, device=DEVICE)
            else:
                hidden_state = hidden_state.detach()


            laplacian       = calculate_laplacian(adj)
            hidden_state, _ = model_final.tgcn_cell(snap.x, hidden_state, laplacian)
            logits          = model_final.classifier(hidden_state).squeeze(1)
            loss            = criterion_final(logits, snap.y.float())
            window_loss    += loss

        window_loss.backward()
        nn.utils.clip_grad_norm_(model_final.parameters(), max_norm=1.0)
        optimizer_final.step()
        epoch_loss += window_loss.item()

    avg_train_loss = epoch_loss / len(windows_final)
    val_loss       = compute_val_loss(model_final, snapshots_val, criterion_final, DEVICE)
    val_metrics    = evaluate(model_final, snapshots_val, DEVICE)
    current_lr     = optimizer_final.param_groups[0]['lr']
    scheduler_final.step(val_loss)

    history_final['train_loss'].append(avg_train_loss)
    history_final['val_loss'].append(val_loss)
    history_final['val_auroc'].append(val_metrics['auroc'])
    history_final['val_auprc'].append(val_metrics['auprc'])
    history_final['lr'].append(current_lr)

    if val_loss < best_loss_final:
        best_loss_final  = val_loss
        best_epoch_final = epoch
        best_wts_final   = copy.deepcopy(model_final.state_dict())
        no_improv_final  = 0
    else:
        no_improv_final += 1

    if epoch % 25 == 0 or epoch == 1:
        elapsed = time.perf_counter() - t0
        print(
            f'Epoch {epoch:4d}/{MAX_EPOCHS_FINAL} | '
            f'train={avg_train_loss:.4f} | '
            f'val_loss={val_loss:.4f} | '
            f'auroc={val_metrics["auroc"]:.4f} | '
            f'auprc={val_metrics["auprc"]:.4f} | '
            f'lr={current_lr:.6f} | '
            f'elapsed={elapsed:.0f}s'
        )

    if no_improv_final >= PATIENCE_FINAL:
        print(f'\nEarly stopping en epoch {epoch}. '
              f'Mejor epoch: {best_epoch_final} (val_loss={best_loss_final:.4f})')
        break

model_final.load_state_dict(best_wts_final)
print(f'\nModelo final. Mejor epoch: {best_epoch_final} | val_loss: {best_loss_final:.4f}')

Mejor configuración:
  hidden_dim: 64.0
  lr: 0.001
  weight_decay: 0.0001
  pw_factor: 0.25
  W: 8.0
  pw: 391.01229508196724
  best_val_loss: 0.0022870163181020566
  val_auroc: 0.9999335658528483
  val_auprc: 0.41666666666666663
Epoch    1/400 | train=4.6892 | val_loss=0.2291 | auroc=0.9991 | auprc=0.0542 | lr=0.001000 | elapsed=2s
Epoch   25/400 | train=0.5190 | val_loss=0.0131 | auroc=0.9999 | auprc=0.3667 | lr=0.001000 | elapsed=64s
Epoch   50/400 | train=0.3003 | val_loss=0.0077 | auroc=0.9999 | auprc=0.4500 | lr=0.001000 | elapsed=128s
Epoch   75/400 | train=0.1310 | val_loss=0.0049 | auroc=0.9999 | auprc=0.4167 | lr=0.001000 | elapsed=194s
Epoch  100/400 | train=0.0560 | val_loss=0.0038 | auroc=0.9999 | auprc=0.3250 | lr=0.001000 | elapsed=258s
Epoch  125/400 | train=0.0321 | val_loss=0.0036 | auroc=0.9999 | auprc=0.3250 | lr=0.001000 | elapsed=322s
Epoch  150/400 | train=0.0250 | val_loss=0.0039 | auroc=0.9999 | auprc=0.3667 | lr=0.000250 | elapsed=387s

Early stopping en epoc

Lo que está bien

train_loss desciende de 4.69 a 0.025, convergencia clara.
val_loss mejora progresivamente: 0.2291 → 0.0034, mejor epoch 120.
ReduceLROnPlateau actúa entre epoch 125 y 150: lr baja de 1e-3 a 2.5e-4 (dos reducciones: 1e-3 → 5e-4 → 2.5e-4).
Early stopping correcto, para en 150 con mejor epoch 120.

Un punto de atención
val_loss del mejor epoch (0.0034) es peor que el val_loss reportado en el grid search para el mismo combo (0.0023). La diferencia se explica porque en el grid search el early stopping usa PATIENCE=20 y MAX_EPOCHS=200, mientras que aquí PATIENCE=30 y MAX_EPOCHS=400. El modelo entrenó más tiempo y el scheduler redujo el lr antes de que el grid search lo hubiera hecho, llegando a un mínimo diferente. No es un problema, el modelo final con 0.0034 es perfectamente válido.
val_auprc oscila entre 0.32 y 0.45 a lo largo del entrenamiento. Con 2 positivos esto es esperado: detectar 0, 1 o 2 positivos cambia la métrica drásticamente. No es señal de inestabilidad del modelo.

In [19]:
# ============================================================
# CELDA 12 — Guardado del modelo y artefactos
# ============================================================
import os
import json
os.makedirs('models_checkpoints', exist_ok=True)

opt_threshold, opt_f1 = find_best_threshold(model_final, snapshots_val, DEVICE)
final_metrics = evaluate(model_final, snapshots_val, DEVICE, threshold=opt_threshold)

print(f'Métricas finales sobre validación (umbral={opt_threshold:.3f}):')
for k, v in final_metrics.items():
    print(f'  {k}: {v:.4f}')

torch.save(model_final.state_dict(), 'models_checkpoints/tgcn_best.pt')
print('Pesos guardados en models_checkpoints/tgcn_best.pt')

config = {
    'input_dim'    : INPUT_DIM,
    'hidden_dim'   : int(best_config['hidden_dim']),
    'lr'           : best_config['lr'],
    'weight_decay' : best_config['weight_decay'],
    'pos_weight'   : best_config['pw'],
    'best_epoch'   : best_epoch_final,
    'val_auprc'    : final_metrics['auprc'],
    'val_f1'       : final_metrics['f1'],
    'opt_threshold': opt_threshold,
    'frontera_val' : FRONTERA_VAL,
    'seed'         : SEED,
}

with open('models_checkpoints/tgcn_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Configuración guardada en models_checkpoints/tgcn_config.json')
print('\nArtefactos listos para notebook 07 (evaluación final).')

Métricas finales sobre validación (umbral=0.940):
  auroc: 0.9999
  auprc: 0.3250
  f1: 0.5000
  precision: 0.3333
  recall: 1.0000
Pesos guardados en models_checkpoints/tgcn_best.pt
Configuración guardada en models_checkpoints/tgcn_config.json

Artefactos listos para notebook 07 (evaluación final).


Umbral=0.940
Muy alto, lo que es esperable con desbalanceo extremo. El modelo asigna probabilidades bajas a casi todo y solo supera 0.94 en los casos más claros. Esto es una consecuencia directa de pw_factor=0.25: el pos_weight suavizado no empuja los logits de positivos tan agresivamente como el ratio bruto.
Métricas

recall=1.0 significa que detecta los 2 positivos de val. Con solo 2 positivos esto es el escenario óptimo posible.
precision=0.333 significa que junto a los 2 verdaderos positivos hay 4 falsos positivos (2/6 = 0.333). Con umbral tan alto es un resultado razonable.
f1=0.50 con 2 positivos y umbral optimizado sobre val es el techo práctico en este conjunto.

Advertencia importante para la memoria
Estas métricas están optimizadas sobre val, que tiene solo 2 positivos. No son generalizables. El umbral 0.940 puede ser demasiado conservador o demasiado permisivo en el periodo de evaluación con 33 positivos. En el notebook 07 habrá que reoptimizar el umbral sobre el conjunto de evaluación o usar directamente AUPRC/AUROC como métricas principales sin depender del umbral.
Estado actual
Los artefactos están guardados. El T-GCN está listo. El siguiente paso es la extracción de e_rel para todos los snapshots de desarrollo.

## 13. Extracción de embeddings relacionales

Una vez entrenado el T-GCN con criterio supervisado (BCE ponderada), los
parámetros de la red se congelan. El modelo deja de actuar como clasificador
y pasa a funcionar como **encoder relacional**: una función determinista
$f_{\theta^*}: (G^t, h^{t-1}) \rightarrow h^t$ que mapea el estado del grafo
en $t$ y el estado oculto acumulado hasta $t-1$ a un embedding
$e_{\text{rel}}^{i,t} = h_i^t \in \mathbb{R}^{d_h}$ por nodo.

Formalmente, para cada banco $i$ activo en el trimestre $t$:

$$
e_{\text{rel}}^{i,t} = h_i^t = \text{TGCNCell}_{\theta^*}(x_i^t,\, h_i^{t-1},\, \hat{L}^t)
$$

donde $\theta^*$ son los pesos óptimos identificados en el grid search,
$x_i^t \in \mathbb{R}^{204}$ son los atributos nodales del snapshot $t$,
y $\hat{L}^t$ es el Laplaciano normalizado del grafo de holdings en $t$.

La extracción se realiza en modo inferencia (`torch.no_grad()`) con
**alineación por CERT**: el estado oculto $h_i^{t-1}$ de cada banco se
recupera de un diccionario indexado por su identificador CERT, garantizando
continuidad del estado oculto para nodos persistentes entre trimestres
independientemente de cambios en $|V^t|$. Los bancos que aparecen por
primera vez en $t$ se inicializan a $h_i^{t-1} = \mathbf{0}$.

El resultado es un conjunto de embeddings
$\{e_{\text{rel}}^{i,t}\}_{i \in V^t,\, t \in \mathcal{T}_{\text{dev}}}$
sobre los 23 trimestres del bloque de desarrollo, con la misma indexación
(CERT, period) que los embeddings tabulares $e_{\text{tab}}^{i,t} \in
\mathbb{R}^{192}$ producidos por TabPFN. Esta alineación permite la
concatenación directa:

$$
e_{\text{hybrid}}^{i,t} = \left[ e_{\text{tab}}^{i,t} \;\|\; e_{\text{rel}}^{i,t} \right]
\in \mathbb{R}^{192 + d_h}
$$

que es el input del MLP de fusión en el notebook siguiente.

La señal supervisada usada durante el entrenamiento del T-GCN (BCE con
etiquetas de quiebra) no tenía como objetivo producir un clasificador
óptimo sino orientar el espacio latente $\mathbb{R}^{d_h}$ hacia
representaciones donde bancos con perfil de riesgo relacional similar
queden próximos. Las métricas AUROC y AUPRC del clasificador lineal
provisional son criterios de selección de hiperparámetros del encoder,
no resultados del sistema. La calidad de $e_{\text{rel}}$ solo es
evaluable en el pipeline completo, tras la fusión y la evaluación final
sobre el bloque de evaluación.

In [20]:
# ============================================================
# CELDA 13 — Extracción de embeddings relacionales e_rel
# ============================================================
from pathlib import Path
import pandas as pd

# Cargar snapshots de desarrollo
snapshots_dev = GraphBuilder.load_snapshots('snapshots_desarrollo')
print(f'Snapshots de desarrollo cargados : {len(snapshots_dev)}')
print(f'Rango temporal : {snapshots_dev[0].period} → {snapshots_dev[-1].period}')

# Congelar modelo
model_final.eval()
for param in model_final.parameters():
    param.requires_grad = False

# Forward pass con alineación por CERT — extracción de hidden_state
records        = []
cert_to_hidden = {}

with torch.no_grad():
    for snap in snapshots_dev:
        snap      = snapshot_to_device(snap, DEVICE)
        adj       = compute_adj_dense(snap)
        cert_list = list(snap.cert)

        hidden_state = _build_hidden_from_dict(
            cert_list, cert_to_hidden, model_final.hidden_dim, DEVICE
        )
        laplacian       = calculate_laplacian(adj)
        hidden_state, _ = model_final.tgcn_cell(snap.x, hidden_state, laplacian)
        _update_hidden_dict(cert_list, hidden_state, cert_to_hidden)

        # e_rel = hidden_state ANTES del classifier — (N, hidden_dim)
        e_rel = hidden_state.cpu().numpy()
        y     = snap.y.cpu().numpy()

        for idx, cert in enumerate(cert_list):
            row = {'CERT': cert, 'period': snap.period}
            row.update({f'e_rel_{j}': e_rel[idx, j]
                        for j in range(e_rel.shape[1])})
            row['y'] = int(y[idx])
            records.append(row)

# Guardar
Path('embeddings/emb_rel').mkdir(parents=True, exist_ok=True)

df_erel = pd.DataFrame(records)
df_erel.to_parquet('embeddings/emb_rel/erel_desarrollo.parquet', index=False)

print(f'\nShape                : {df_erel.shape}')
print(f'Periodos             : {df_erel["period"].nunique()}')
print(f'Positivos / total    : {df_erel["y"].sum()} / {len(df_erel)}')
print(f'Dimensión e_rel      : {model_final.hidden_dim}')
print('Guardado en embeddings/emb_rel/erel_desarrollo.parquet')

Cargados 23 snapshots desde snapshots_desarrollo
Snapshots de desarrollo cargados : 23
Rango temporal : 2016Q2 → 2021Q4

Shape                : (125575, 67)
Periodos             : 23
Positivos / total    : 63 / 125575
Dimensión e_rel      : 64
Guardado en embeddings/emb_rel/erel_desarrollo.parquet


1. Dimensiones correctas
Shape (125575, 67) = 125575 nodos × (CERT + period + 64 dims + y). Correcto.
2. Positivos correctos
63 positivos sobre 23 trimestres coincide exactamente con el total documentado en CONTEXT08 (61 train + 2 val). Correcto.
3. Que los embeddings no son constantes ni degenerados
4. Que positivos y negativos tienen distribuciones distintas

In [22]:
import numpy as np
e_rel_cols = [c for c in df_erel.columns if c.startswith('e_rel_')]
vals = df_erel[e_rel_cols].values

print(f'Media global  : {vals.mean():.6f}')
print(f'Std global    : {vals.std():.6f}')
print(f'Min / Max     : {vals.min():.6f} / {vals.max():.6f}')
print(f'Dims con std=0: {(vals.std(axis=0) == 0).sum()}')


pos = df_erel[df_erel['y'] == 1][e_rel_cols].values
neg = df_erel[df_erel['y'] == 0][e_rel_cols].values

print(f'Media positivos : {pos.mean():.6f}')
print(f'Media negativos : {neg.mean():.6f}')
print(f'Std positivos   : {pos.std():.6f}')
print(f'Std negativos   : {neg.std():.6f}')

Media global  : 0.049390
Std global    : 0.882876
Min / Max     : -1.000000 / 1.000000
Dims con std=0: 0
Media positivos : -0.132684
Media negativos : 0.049482
Std positivos   : 0.853028
Std negativos   : 0.882881


Análisis
Rango [-1, 1] con std=0.88
Esperado. La salida de la celda GRU es tanh para el candidato y combinación convexa para la actualización, por lo que los valores están acotados en [-1, 1] por construcción. Ninguna dimensión constante confirma que todas las 64 dims aportan varianza.
Separación positivos vs negativos
Media positivos = -0.133 vs media negativos = +0.049. Diferencia pequeña en valor absoluto pero consistente: la BCE supervisada ha orientado el espacio latente de forma que los bancos quebrados tienen en promedio embeddings desplazados respecto a los sanos. Con solo 63 positivos sobre 125.575 nodos la separación no puede ser dramática, pero existe y es la señal que el MLP de fusión explotará.
Std similar en ambos grupos
0.853 vs 0.883. El encoder no ha colapsado los positivos a un punto sino que mantiene varianza interna, lo que significa que hay estructura dentro de los quebrados, no solo una única representación promedio.
Conclusión
Los embeddings son válidos para pasar al MLP de fusión. La verificación es todo lo que puedes hacer antes del híbrido.